# Using pretrained models (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [3]:
!pip install torch transformers accelerate
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)


In [4]:
# Check GPU first
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not found! Go to Runtime > Change runtime type > T4 GPU.")

# Use the official Colab-optimized installer
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ejht9jc1/unsloth_06ec96ab29c748578260c90a414ab15e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ejht9jc1/unsloth_06ec96ab29c748578260c90a414ab15e
  Resolved https://github.com/unslothai/unsloth.git to commit 050240b27a0c0cba0f33a1720e14dde8f48a6eb2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load the tokenizer and model
# torch_dtype="auto" automatically chooses between float16/bfloat16/float32
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define your conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain vibe coding in one sentence."}
]

# 3. Apply the chat template (adds special tokens like <|im_start|>)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 4. Tokenize and generate
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7
)

# 5. Decode the output (removing the input prompt)
response_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vibe coding is the practice of creating code that feels intuitive, enjoyable to work with, and aligns with your personal style and preferences.


In [9]:

# 1. INSTALL DEPENDENCIES (Colab specific)


import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import json

# 2. CREATE THE DATASET FILE (Synthetic data we discussed)
data = [
    {"conversations": [{"role": "user", "content": "I need a high-resolution 3D render of a futuristic cyberpunk city."}, {"role": "assistant", "content": '{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "false"}}}'}]},
    {"conversations": [{"role": "user", "content": "What are the latest breaking news headlines?"}, {"role": "assistant", "content": '{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "true"}}}'}]},
    {"conversations": [{"role": "user", "content": "Search for cat food prices and show me a photo of a cat."}, {"role": "assistant", "content": '{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "true"}}}'}]},
    {"conversations": [{"role": "user", "content": "Refactor this Python function: def add(a,b): return a+b"}, {"role": "assistant", "content": '{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "false"}}}'}]},
    {"conversations": [{"role": "user", "content": "Visualize a minimalist dashboard for a dev-ops tool."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Check the current status of the AWS us-east-1 region."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "I need a concept art piece of a forest spirit and can you find what folklore it might be from?"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "How do I implement a binary search tree in TypeScript?"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Create a diagram showing the architecture of a RAG pipeline."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Who is currently leading the 2026 World Cup qualifiers?"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Generate a UI mockup for a music streaming app and find the hex codes for Spotify's branding."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Explain the difference between a shallow copy and a deep copy in JS."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "What's the weather like in London right now?"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Paint me a digital portrait of a Victorian-era astronaut."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Look up the docs for the latest FastAPI release and summarize the breaking changes."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Give me a logo for my startup called 'CloudByte' and see if the domain is available."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Write a regex to match email addresses."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Show me an icon set for an iOS weather app."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Find the release date for the next Marvel movie."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Check if there's any active fire in the Napa Valley area via satellite news."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Illustrate a knight fighting a dragon in a watercolor style."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
{"conversations": [{"role": "user", "content": "Get the current Bitcoin price and generate a social media banner for a crypto-exchange."}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"true\"}, \"handle_web_search\": {\"intent\": \"true\"}}}"}]},
{"conversations": [{"role": "user", "content": "Debug this SQL query: SELECT * FROM users WHERE id = 'none'"}, {"role": "assistant", "content": "{\"function\": {\"handle_generate_image\": {\"intent\": \"false\"}, \"handle_web_search\": {\"intent\": \"false\"}}}"}]},
]

with open("training_data.jsonl", "w") as f:
    for entry in data:
        f.write(json.dumps(entry) + "\n")

# 3. LOAD MODEL & TOKENIZER
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 4. SETUP LoRA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 5. DATA PREPARATION
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

def format_prompts(examples):
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in examples["conversations"]]
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

# 6. TRAIN
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # Quick training for demo
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)
trainer.train()

# 7. INFERENCE TEST
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Check the news and draw a robot."}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64)
print("\n--- TEST RESULT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1])

# 8. SAVE
model.save_pretrained("qwen_vibe_router")
tokenizer.save_pretrained("qwen_vibe_router")

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/24 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 14 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,4.257104
2,4.340327
3,4.063103
4,3.478866
5,3.018981
6,2.676934
7,2.367281
8,1.986356
9,1.725511
10,1.467437



--- TEST RESULT ---
{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "false"}}}


('qwen_vibe_router/tokenizer_config.json',
 'qwen_vibe_router/chat_template.jinja',
 'qwen_vibe_router/tokenizer.json')

In [6]:
from unsloth import FastLanguageModel
import torch

# 1. Load the model and tokenizer
# Point 'model_name' to the folder where you saved it
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen_vibe_router",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Switch to fast inference mode
FastLanguageModel.for_inference(model)

# 3. Use it in your code
def get_intent(user_input):
    messages = [{"role": "user", "content": user_input}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=100)
    # Extract only the assistant's JSON response
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("assistant\n")[-1]

# Example test
print(get_intent("Search for the latest NVIDIA stock price"))

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "true"}}}


In [11]:
print(get_intent("give me a picture of current iran war"))
print(get_intent("123456jhgjhgjh"))
print(get_intent("selfDestruct"))
print(get_intent("what does 8964 mean in Cina ?"))
print(get_intent("im working on art. need a illustration of sorrw "))
print(get_intent("ImWorking on art. need aIllustration of sorrw "))

{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "false"}}}
{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "false"}}}
{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "false"}}}
{"function": {"handle_generate_image": {"intent": "false"}, "handle_web_search": {"intent": "true"}}}
{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "false"}}}
{"function": {"handle_generate_image": {"intent": "true"}, "handle_web_search": {"intent": "false"}}}
